In [0]:
%sql
--Total revenue per category (revenue = quantity × unit_price × (1 - discount_percent/100)) 
SELECT p.category, ROUND(SUM(oi.quantity* oi.unit_price* (1 - oi.discount_percent / 100)),2) AS total_revenue
FROM order_items oi JOIN products p ON oi.product_id = p.product_id GROUP BY p.category
ORDER BY total_revenue DESC;

category,total_revenue
Electronics,1.6653330118E8
Home,6.006038479E7
Sports,1.529010091E7
Beauty,1.369129807E7
Clothing,5567218.51
Books,2634679.67


In [0]:
%sql
--Top 10 customers by total order value 
SELECT o.customer_id, ROUND(SUM(oi.quantity * oi.unit_price * (1 - oi.discount_percent / 100)), 2) AS total_order_value
FROM orders o JOIN order_items oi ON o.order_id = oi.order_id WHERE o.customer_id IS NOT NULL GROUP BY o.customer_id
ORDER BY total_order_value DESC;


customer_id,total_order_value
248,1256409.24
15,1211641.0
695,1189463.41
771,1080227.64
372,1057590.69
501,1034852.92
315,1024323.44
450,993905.91
364,970581.78
561,940514.75


In [0]:
%sql
--Month-wise order count for the last 12 months 
WITH max_date AS (SELECT MAX(order_date) AS latest_date FROM orders)SELECT DATE_FORMAT(o.order_date, 'yyyy-MM') AS order_month,
COUNT(*) AS order_count FROM orders o CROSS JOIN max_date m WHERE o.order_date >= ADD_MONTHS(m.latest_date, -11)
GROUP BY DATE_FORMAT(o.order_date, 'yyyy-MM') ORDER BY order_month;

order_month,order_count
2025-09,143
2025-10,216
2025-11,210
2025-12,192
2026-01,221
2026-02,208
2026-03,214
2026-04,199
2026-05,213
2026-06,203


In [0]:
%sql
--Find customers who placed orders but never had any item delivered 
SELECT o.customer_id, COUNT(DISTINCT o.order_id) AS total_orders FROM orders o WHERE o.customer_id IS NOT NULL
GROUP BY o.customer_id HAVING SUM( CASE WHEN o.status = 'DELIVERED' THEN 1 ELSE 0 END) = 0
ORDER BY total_orders DESC;

customer_id,total_orders
989,11
100,9
924,9
324,9
784,9
797,9
517,8
171,8
354,8
675,8


In [0]:
%sql
--Products that were ordered but had more returns than purchases 
SELECT oi.product_id, p.product_name, SUM(CASE WHEN oi.quantity > 0 THEN oi.quantity ELSE 0 END) AS total_purchased,
SUM(CASE WHEN oi.quantity < 0 THEN ABS(oi.quantity) ELSE 0 END) AS total_returned FROM order_items oi
JOIN products p ON oi.product_id = p.product_id GROUP BY oi.product_id, p.product_name
HAVING total_returned > total_purchased ORDER BY total_returned DESC;


product_id,product_name,total_purchased,total_returned


In [0]:
%sql
--Calculate the return rate (returned items / total items) per category 
SELECT p.category, SUM( CASE WHEN oi.quantity < 0 THEN ABS(oi.quantity) ELSE 0 END ) AS returned_items, SUM(ABS(oi.quantity)) 
AS total_items, ROUND(100.0 *SUM(CASE WHEN oi.quantity < 0 THEN ABS(oi.quantity) ELSE 0 END)/ NULLIF(SUM(ABS(oi.quantity)), 0),2)
AS return_rate_percent FROM order_items oi JOIN products p ON oi.product_id = p.product_id GROUP BY p.category
ORDER BY return_rate_percent DESC;

category,returned_items,total_items,return_rate_percent
Sports,207,5958,3.47
Electronics,247,7340,3.37
Clothing,152,5167,2.94
Beauty,318,10936,2.91
Home,231,8437,2.74
Books,183,7472,2.45
